# NB04 — Agentic RAG Pipeline (Method 2)
**MSK | Goel Lab** · LLM Systematic Review · Method Comparison

**Purpose**: Run the LangGraph-based agentic RAG agent against the standardised query set.  
Vector store: **Pinecone** (serverless). Source documents are returned alongside every answer using two complementary LangChain patterns from the [official RAG docs](https://docs.langchain.com/oss/python/langchain/rag#returning-source-documents).

**Stack**: LangChain 0.3.x · `langchain-pinecone` · GPT-4o (T=0) · DeepEval

**Source-document patterns used**
| Pattern | Where |
|---------|-------|
| `RunnableParallel` + `RunnablePassthrough` → returns `{context, question, answer}` | Standalone RAG chain test |
| `RetrieveDocumentsMiddleware` (`State.context: list[Document]`) | Agentic loop — sources logged per iteration |

**Outputs**
- `data/processed/agentic_rag_results.csv`
- `experiments/<run_id>.json` (×3 runs)

In [ ]:
import sys
sys.path.insert(0, '..')

import os
import json
import time
from pathlib import Path
from typing import Any

from dotenv import load_dotenv
load_dotenv()

# ── Pinecone ──────────────────────────────────────────────────────────────────
from pinecone import Pinecone, ServerlessSpec
from langchain_pinecone import PineconeVectorStore

# ── LangChain ─────────────────────────────────────────────────────────────────
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_community.document_loaders import PubMedLoader, PyPDFDirectoryLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough, RunnableParallel

# ── DeepEval ──────────────────────────────────────────────────────────────────
from deepeval.test_case import LLMTestCase
from deepeval.metrics import FaithfulnessMetric

# ── Project ───────────────────────────────────────────────────────────────────
from src.utils.io import data_dir, project_root, generate_run_id, log_experiment
from src.methods.agentic_rag.agent import AgenticRAGAgent

import pandas as pd

# ── Config ────────────────────────────────────────────────────────────────────
OPENAI_API_KEY   = os.environ["OPENAI_API_KEY"]
PINECONE_API_KEY = os.environ["PINECONE_API_KEY"]
INDEX_NAME       = os.environ.get("PINECONE_INDEX_NAME", "llm-sysrev-corpus")
EMBED_MODEL      = "text-embedding-3-small"   # 1536-dim
EMBED_DIM        = 1536
LLM_MODEL        = "gpt-4o"
TOP_K            = 10
CHUNK_SIZE       = 800
CHUNK_OVERLAP    = 150

embeddings = OpenAIEmbeddings(model=EMBED_MODEL, openai_api_key=OPENAI_API_KEY)
llm        = ChatOpenAI(model=LLM_MODEL, temperature=0, openai_api_key=OPENAI_API_KEY)

print("Imports OK.")

## 1. Connect to Pinecone

Creates the index if it doesn't exist (serverless, `us-east-1`).  
If the index already exists from a previous run, this is a no-op — it just loads it.

In [ ]:
pc = Pinecone(api_key=PINECONE_API_KEY)

existing = [idx.name for idx in pc.list_indexes()]
if INDEX_NAME not in existing:
    pc.create_index(
        name=INDEX_NAME,
        dimension=EMBED_DIM,
        metric="cosine",
        spec=ServerlessSpec(cloud="aws", region="us-east-1"),
    )
    print(f"[pinecone] Index '{INDEX_NAME}' created (serverless, us-east-1).")
else:
    print(f"[pinecone] Index '{INDEX_NAME}' already exists — loading.")

index = pc.Index(INDEX_NAME)
stats = index.describe_index_stats()
print(f"[pinecone] Vectors in index: {stats['total_vector_count']}")

## 2. Index corpus into Pinecone  *(run once)*

Skip to **Section 3** if the index already contains your vectors.

In [ ]:
## 2a. Load + chunk documents (same splitter as NB03 for comparability)

_SECTION_SEPS = [
    "\nAbstract\n", "\nINTRODUCTION\n", "\nIntroduction\n",
    "\nMETHODS\n",  "\nMethods\n",       "\nMaterials and Methods\n",
    "\nRESULTS\n",  "\nResults\n",
    "\nDISCUSSION\n", "\nDiscussion\n",
    "\nCONCLUSION\n", "\nConclusion\n",
    "\nREFERENCES\n", "\n\n", "\n", ". ", " ",
]

splitter = RecursiveCharacterTextSplitter(
    separators=_SECTION_SEPS,
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
    add_start_index=True,
)

# ── Load from PubMed (swap for PDF/JSON loader if needed) ────────────────────
PUBMED_QUERY = (
    '("large language model"[tiab] OR "LLM"[tiab] OR "GPT"[tiab]) '
    'AND ("feature extraction"[tiab] OR "information extraction"[tiab]) '
    'AND ("radiology report"[tiab] OR "pathology report"[tiab] OR "clinical note"[tiab]) '
    'AND ("2018/01/01"[PDAT] : "2026/12/31"[PDAT])'
)
loader = PubMedLoader(query=PUBMED_QUERY, load_max_docs=500)
docs   = loader.load()
chunks = splitter.split_documents(docs)
print(f"[ingest] {len(docs)} docs → {len(chunks)} chunks")

## 2b. Upsert into Pinecone
vectorstore = PineconeVectorStore.from_documents(
    documents=chunks,
    embedding=embeddings,
    index_name=INDEX_NAME,
)
print(f"[pinecone] Upsert complete. Re-run describe_index_stats() to confirm vector count.")

## 3. Source-returning RAG chain (LCEL)

**Pattern** — `RunnableParallel` wraps the retriever so the chain returns both the answer **and** the source `Document` objects in one call:

```python
chain_with_source = RunnableParallel(
    {"context": retriever, "question": RunnablePassthrough()}
).assign(answer=answer_chain)

result = chain_with_source.invoke("...")
# result["context"]  → list[Document]
# result["answer"]   → str
# result["question"] → str
```

Ref: [docs.langchain.com/…/rag#returning-source-documents](https://docs.langchain.com/oss/python/langchain/rag#returning-source-documents)

In [ ]:
# Load existing Pinecone index (skip section 2 on subsequent runs)
vectorstore = PineconeVectorStore(index_name=INDEX_NAME, embedding=embeddings)
retriever   = vectorstore.as_retriever(search_type="similarity", search_kwargs={"k": TOP_K})

_REVIEW_PROMPT = ChatPromptTemplate.from_messages([
    ("system", (
        "You are a systematic review researcher.\n"
        "Answer using ONLY the retrieved study excerpts in <context>.\n"
        "For each claim cite the source title and year in parentheses.\n"
        "If the evidence is insufficient say so — do not speculate.\n\n"
        "Structure: 1) Summary  2) Key findings (cited)  3) Gaps  4) PICO\n\n"
        "<context>\n{context}\n</context>"
    )),
    ("human", "{question}"),
])

def _format_docs(docs: list[Document]) -> str:
    return "\n\n".join(
        f"[{d.metadata.get('title', d.metadata.get('source', '?'))} "
        f"({d.metadata.get('year', 'n.d.')})] {d.page_content}"
        for d in docs
    )

# Answer chain (context already formatted as str)
_answer_chain = (
    RunnablePassthrough.assign(context=lambda x: _format_docs(x["context"]))
    | _REVIEW_PROMPT
    | llm
    | StrOutputParser()
)

# ── Source-returning chain ────────────────────────────────────────────────────
# Returns {"context": list[Document], "question": str, "answer": str}
rag_chain_with_source = RunnableParallel(
    {"context": retriever, "question": RunnablePassthrough()}
).assign(answer=_answer_chain)

# Quick test
TEST_Q = (
    "What LLM optimization strategies have been applied to "
    "radiology report feature extraction?"
)
result = rag_chain_with_source.invoke(TEST_Q)

print("=" * 70)
print("ANSWER:\n")
print(result["answer"])
print("\n" + "=" * 70)
print(f"\nSOURCE DOCUMENTS  ({len(result['context'])} retrieved)\n")
for i, doc in enumerate(result["context"], 1):
    title = doc.metadata.get("title", doc.metadata.get("source", "?"))
    year  = doc.metadata.get("year", "n.d.")
    print(f"[{i}] {title} ({year})")
    print(f"    {doc.page_content[:200]}…\n")

## 4. RetrieveDocumentsMiddleware — source tracking inside the agent loop

For the agentic pipeline, sources need to be captured **per iteration** so we know which retrieval step first surfaced a useful chunk.  

The pattern below follows the LangChain docs (`AgentMiddleware` + custom `State`):  
- `State.context` accumulates all `Document` objects retrieved across iterations  
- `before_model` hook fires before every LLM call — retrieves fresh docs, appends to state, augments the message with context  

```python
class State(AgentState):
    context: list[Document]          # grows across iterations

class RetrieveDocumentsMiddleware(AgentMiddleware[State]):
    state_schema = State
    def before_model(self, state, ...) -> dict:
        docs = vector_store.similarity_search(last_message)
        return {"messages": [...augmented...], "context": accumulated_docs}
```

In [ ]:
from langchain_core.documents import Document as LCDocument

# ── LangChain retriever wrapper for AgenticRAGAgent ───────────────────────────
class PineconeRetrieverWrapper:
    """Adapts the LangChain Pinecone retriever to the agent's vector_pipeline interface.
    Tracks per-iteration source documents for provenance logging."""

    def __init__(self, retriever):
        self._retriever = retriever
        self.iteration_sources: list[dict] = []   # populated during agent run

    def retrieve(self, query: str, top_k: int = TOP_K) -> pd.DataFrame:
        docs = self._retriever.invoke(query)[:top_k]
        sources = [
            {
                "title":   d.metadata.get("title", d.metadata.get("source", "?")),
                "year":    d.metadata.get("year", "n.d."),
                "excerpt": d.page_content[:200],
                "query":   query,
            }
            for d in docs
        ]
        self.iteration_sources.append({"query": query, "sources": sources})
        return pd.DataFrame([
            {"doc_id": d.metadata.get("source", ""), "text": d.page_content}
            for d in docs
        ])

    def reset(self):
        self.iteration_sources = []

retriever_wrapper = PineconeRetrieverWrapper(retriever)
agent = AgenticRAGAgent(vector_pipeline=retriever_wrapper)
print("Agent ready — Pinecone retriever wired.")

## 5. Run 3× for reproducibility assessment

Runs the agent three times with the same question.  
Between runs `retriever_wrapper.reset()` clears the iteration-source log so each run is tracked independently.

In [ ]:
REVIEW_QUESTION = (
    "What LLM optimization strategies (prompt engineering, fine-tuning, RAG, "
    "chain-of-thought, instruction tuning) have been proposed or evaluated for "
    "extracting structured clinical features from radiology and pathology reports?"
)

run_results = []

for i in range(3):
    retriever_wrapper.reset()
    run_id = generate_run_id("agentic_rag")
    t0 = time.perf_counter()

    result = agent.run(question=REVIEW_QUESTION, run_id=run_id)
    result["time_seconds"] = round(time.perf_counter() - t0, 1)
    result["run_id"] = run_id

    # Attach per-iteration source provenance from the wrapper
    result["iteration_sources"] = retriever_wrapper.iteration_sources.copy()

    run_results.append(result)
    log_experiment(result)

    print(f"\nRun {i+1}  [{run_id}]")
    print(f"  docs retrieved : {result.get('n_retrieved', '?')}")
    print(f"  iterations     : {result.get('n_iterations', '?')}")
    print(f"  time           : {result['time_seconds']}s")
    print(f"  source log     : {len(result['iteration_sources'])} retrieval steps")

    # Print sources from first iteration of this run
    if result["iteration_sources"]:
        first = result["iteration_sources"][0]
        print(f"\n  Sub-query: \"{first['query'][:80]}…\"")
        print(f"  Top sources:")
        for s in first["sources"][:3]:
            print(f"    • {s['title']} ({s['year']})")

## 6. Jaccard reproducibility across 3 runs

Measures set overlap of retrieved `doc_id`s between every pair of runs.  
Values close to 1.0 = highly stable retrieval; values below 0.6 suggest sensitivity to LLM stochasticity.

In [ ]:
doc_id_sets = [set(r.get("retrieved_doc_ids", [])) for r in run_results]

jaccards = []
for i in range(len(doc_id_sets)):
    for j in range(i + 1, len(doc_id_sets)):
        inter = len(doc_id_sets[i] & doc_id_sets[j])
        union = len(doc_id_sets[i] | doc_id_sets[j])
        j_sim = inter / union if union else 0.0
        jaccards.append(j_sim)
        print(f"Run {i+1} vs Run {j+1}: Jaccard = {j_sim:.3f}  "
              f"(|∩|={inter}  |∪|={union})")

mean_j = sum(jaccards) / len(jaccards) if jaccards else 0.0
print(f"\nMean Jaccard across 3 runs: {mean_j:.3f}")
print("Interpretation:", "✓ Stable" if mean_j >= 0.6 else "⚠ Variable — check LLM temperature or top_k")

## 7. Save results + source provenance

In [ ]:
summary_rows = []
for r in run_results:
    # Flatten all sources across iterations for this run
    all_sources = [
        s
        for step in r.get("iteration_sources", [])
        for s in step["sources"]
    ]
    unique_titles = list({s["title"] for s in all_sources})

    summary_rows.append({
        "run_id":          r["run_id"],
        "n_retrieved":     r.get("n_retrieved", len(unique_titles)),
        "n_iterations":    r.get("n_iterations", len(r.get("iteration_sources", []))),
        "time_seconds":    r["time_seconds"],
        "mean_jaccard":    mean_j,
        "source_titles":   " | ".join(unique_titles[:10]),   # first 10 for CSV
    })

out_path = data_dir() / "processed" / "agentic_rag_results.csv"
out_path.parent.mkdir(parents=True, exist_ok=True)
pd.DataFrame(summary_rows).to_csv(out_path, index=False)
print(f"Results saved → {out_path}")

# Save full source provenance as JSON for NB06 cross-method comparison
prov_path = project_root() / "reports" / "agentic_rag_provenance.json"
prov_path.parent.mkdir(parents=True, exist_ok=True)
prov_out = [
    {
        "run_id":            r["run_id"],
        "iteration_sources": r.get("iteration_sources", []),
    }
    for r in run_results
]
prov_path.write_text(json.dumps(prov_out, indent=2, default=str))
print(f"Source provenance saved → {prov_path}")